In [2]:
from probability import normal_cdf, inverse_normal_cdf # from ch7
import math, random

#### Coin Flip

In [3]:
def normal_approximation_to_binomial(n,p):
    """ finds mu and sigma corresponding to a Binomial(n,p)"""
    mu = p * n
    sigma = math.sqrt(p * (1-p) * n)
    return mu, sigma

#### Probability a Random Variable that Follows a Normal Distribution Falls Within a Particular Interval

In [4]:
normal_probability_below = normal_cdf

def normal_probability_above(lo, mu=0, sigma=1):
    return 1 - normal_cdf(lo, mu, sigma)

def normal_probability_between(lo, hi, mu=0, sigma=1):
    return normal_cdf(hi, mu, sigma) - normal_cdf(lo, mu, sigma)

def normal_probability_outside(lo, hi, mu=0, sigma=1):
    return 1 - normal_probability_between(lo, hi, mu, sigma)

#### Normal Upper, Lower and Two-Sided Bounds

In [5]:
def normal_upper_bound(probability, mu=0, sigma=1):
    """ returns the z for which P(Z <= z) = probability"""
    return inverse_normal_cdf(probability, mu, sigma)

def normal_lower_bound(probability, mu=0, sigma=1):
    """ returns the z for which P(Z >= z) = probability"""
    return inverse_normal_cdf(1 - probability, mu, sigma)

def normal_two_sided_bounds(probability, mu=0, sigma=1):
    """ returns the symmetric (about the mean) bounds
    that contain the specified probability """
    tail_probability = (1 - probability) / 2
    upper_bound = normal_lower_bound(tail_probability, mu, sigma)
    lower_bound = normal_upper_bound(tail_probability, mu, sigma)
    return lower_bound, upper_bound

In [8]:
mu_0, sigma_0 = normal_approximation_to_binomial(1000, 0.5)
print(f'Mu: {mu_0} \nSigma: {sigma_0}')

Mu: 500.0 
Sigma: 15.811388300841896


In [10]:
normal_two_sided_bounds(0.95, mu_0, sigma_0)

(469.01026640487555, 530.9897335951244)

In [12]:
# 95% bounds based on assumptionp is 0.5
lo, hi = normal_two_sided_bounds(0.95, mu_0, sigma_0)
print(f'Lo: {lo} \nHi: {hi}')

Lo: 469.01026640487555 
Hi: 530.9897335951244


In [14]:
# actual mu and sigma based on p=0.55
mu_1, sigma_1 = normal_approximation_to_binomial(1000, 0.55)
print(f'Mu: {mu_1} \nSigma: {sigma_1}')

Mu: 550.0 
Sigma: 15.732132722552274


In [21]:
# a type 2 error means we fail to reject the null hypothesis
# which will happen when X is still in our original interval
type_2_probability = normal_probability_between(lo, hi, mu_1, sigma_1)
print(f'Power: {1 - type_2_probability}')

Power: 0.8865480012953671


In [24]:
# Number is 526 (< 531, since we need more probability in the upper tail
hi = normal_upper_bound(0.95, mu_0, sigma_0)
print(f'Hi: {hi}')

Hi: 526.0073585242053


In [27]:
type_2_probability = normal_probability_below(hi, mu_1, sigma_1)
print(f'Power: {1-type_2_probability}')

Power: 0.9363794803307173


#### P-value

In [28]:
def two_sided_p_value(x, mu=0, sigma=1):
    if x >= mu:
        return 2 * normal_probability_above(x, mu, sigma)
    else:
        return 2 * normal_probability_below(x, mu, sigma)
        

In [29]:
two_sided_p_value(529.5, mu_0, sigma_0)

0.06207721579598835

In [31]:
def count_extreme_values():
    extreme_value_count = 0
    for _ in range(100000):
        num_heads = sum(1 if random.random() < 0.5 else 0
                        for _ in range(1000))
        if num_heads >= 530 or num_heads <= 470:
            extreme_value_count +=1
        return extreme_value_count / 100000
            

In [35]:
upper_p_value = normal_probability_above
lower_p_value = normal_probability_below
print(f'Upper P Value: {upper_p_value(524.5, mu_0, sigma_0)}')
print(f'Lower P Value: {lower_p_value(526.5, mu_0, sigma_0)}')

Upper P Value: 0.06062885772582072
Lower P Value: 0.9531316049114076


#### Confidence Interval

In [37]:
p_hat = 525/1000
mu = p_hat
sigma = math.sqrt(p_hat*(1-p_hat) / 1000)
normal_two_sided_bounds(0.95, mu, sigma)

(0.4940490278129096, 0.5559509721870904)

In [38]:
p_hat = 540/1000
mu = p_hat
sigma = math.sqrt(p_hat*(1-p_hat) / 1000)
normal_two_sided_bounds(0.95, mu, sigma)

(0.5091095927295919, 0.5708904072704082)

#### P-Hacking

In [41]:
def run_experiment():
    """ flip a fair coin 1000 times, True = heads, False = tails"""
    return [random.random() < 0.5 for _ in range(1000)]

def reject_fairness(experiment):
    """ using the 5% significance level"""
    num_heads = len([flip for flip in experiment if flip])
    return num_heads < 469 or num_heads > 531

In [42]:
random.seed(0)
experiments = [run_experiment() for _ in range(1000)]
num_rejections = len([experiment
                      for experiment in experiments
                      if reject_fairness(experiment)])
print(f'Number of rejections: {num_rejections}')

Number of rejections: 46


#### Running an A/B Test

In [43]:
def estimated_parameters(N, n):
    p = n/N
    sigma = math.sqrt(p * (1-p) / N)
    return p, sigma

def a_b_test_statistic(N_A, n_A, N_B, n_B):
    p_A, sigma_A = estimated_parameters(N_A, n_A)
    p_B, sigma_B = estimated_parameters(N_B, n_B)
    return (p_B - p_A) / math.sqrt(sigma_A ** 2 + sigma_B ** 2)

In [44]:
print("A/B testing")
z = a_b_test_statistic(1000, 200, 1000, 180)
print("a_b_test_statistic(1000, 200, 1000, 180)", z)
print("p-value", two_sided_p_value(z))
z = a_b_test_statistic(1000, 200, 1000, 150)
print("a_b_test_statistic(1000, 200, 1000, 150)", z)
print("p-value", two_sided_p_value(z))



A/B testing
a_b_test_statistic(1000, 200, 1000, 180) -1.1403464899034472
p-value 0.254141976542236
a_b_test_statistic(1000, 200, 1000, 150) -2.948839123097944
p-value 0.003189699706216853


#### Bayesian Inference

In [45]:
def B(alpha, beta):
    """ a normalizing constant so that the total probability is 1 """
    return match.gamma(alpha) * math.gamma(beta) / math.gamma(alpha + beta)

def beta_pdf(x, alpha, beta):
    if x < 0 or x > 1:
        return 0
    return x ** (alpha - 1) (1-x) ** (beta - 1) / B(alpha, beta)